In [10]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

In [ ]:
#escribir donde se va a guardar el experimento
mlflow.set_tracking_uri("http://127.0.0.1:8080")

In [21]:
# Sets the current active experiment to the "Apple_Models" experiment and
# returns the Experiment metadata
apple_experiment = mlflow.set_experiment("Apple_Models_3")

# Define a run name for this iteration of training.
# If this is not set, a unique name will be auto-generated for your run.
run_name = "apples_rf_test_3_2"

# Define an artifact path that the model will be saved to.
artifact_path = "rf_apples"


In [8]:
    # 1. Lectura de Datos
direccion="C:\\Users\\Anuar\\Documents\\Maestria Inteligencia artificial\\MLOPs Prueba\\Cookie cutter\\proyecto_mlops_equipo_56_fase_2\\data\\external\\dataset_apples.csv"
data = pd.read_csv(direccion) # Ya no necesitas df_ventas, usa 'data' directamente

In [22]:
# Split the data into features and target and drop irrelevant date field and target field
X = data.drop(columns=["date", "demand"])
y = data["demand"]

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

params = {
    "n_estimators": 100,
    "max_depth": 6,
    "min_samples_split": 10,
    "min_samples_leaf": 4,
    "bootstrap": True,
    "oob_score": False,
    "random_state": 888,
}

# Train the RandomForestRegressor
rf = RandomForestRegressor(**params)

# Fit the model on the training data
rf.fit(X_train, y_train)

# Predict on the validation set
y_pred = rf.predict(X_val)

# Calculate error metrics
mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_pred)

# Assemble the metrics we're going to write into a collection
metrics = {"mae": mae, "mse": mse, "rmse": rmse, "r2": r2}

In [ ]:
# Initiate the MLflow run context
#Esto no guarda un modelo, solo guarda una corrida
#Util para no guardar todos los modelos

with mlflow.start_run(run_name=run_name) as run:
    # Log the parameters used for the model fit
    mlflow.log_params(params)

    # Log the error metrics that were calculated during validation
    mlflow.log_metrics(metrics)

    # Log an instance of the trained model for later use
    mlflow.sklearn.log_model(sk_model=rf, input_example=X_val, name=artifact_path)




c:\Users\Anuar\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run apples_rf_test_3_2 at: http://127.0.0.1:8080/#/experiments/297784242467499909/runs/e9913e0d710941fab45ab10b758356e4
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/297784242467499909


In [ ]:
#Para registrar un modelo se usa: 
 # Log the sklearn model and register as version 1
mlflow.sklearn.log_model(
    sk_model=model,
    name="sklearn-model",
    input_example=X_train,
    registered_model_name="sk-learn-random-forest-reg-model")